In [20]:
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2

# set current directory
os.chdir(os.path.dirname(os.path.abspath(r'C:\Users\Goard Lab\Documents\GitHub\Distance-Coding-Analysis\src')))
# add to the path the folder src
import sys
sys.path.append(r'C:\Users\Goard Lab\Documents\GitHub\Distance-Coding-Analysis\src')
# import modules inside the folder src

from src.DistanceCalibration import *
from src.DistanceProject import DistanceStimulusLoader
from sklearn.mixture import GaussianMixture
from scipy import *
%load_ext autoreload
%autoreload 2
calibration_folder = r'D:\data-distance-coding-project\calibration\240413'
save_path = r'D:\data-distance-coding-project\calibration\240413\all_frames'
video_file = r'D:\data-distance-coding-project\calibration\240413\video_calibration_240413.h264'

# Reading the video

In [2]:
# I want to read every second
cap = cv2.VideoCapture(video_file)
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# Create a VideoCapture object and read from input file
# If the input is the camera, pass 0 instead of the video file name
vidcap = cv2.VideoCapture(video_file)

success,image = vidcap.read()
count = 0
while success:
    
    cv2.imwrite(os.path.join(save_path, "frame%d.jpg" % count), image)     # save frame as JPEG file
        
    success,image = vidcap.read()
    count += 1

# Set parameters with a random frame 

In [147]:
frame = cv2.imread(os.path.join(save_path, "frame405.jpg"))
plt.imshow(frame)

pts_region = get_contour(frame, threshold = 40, show = False, approx = False, single_region = False)

Y = []
for pts in pts_region:
    plt.plot(pts[:,0], pts[:,1])
    M = cv2.moments(pts)
    cX = int(M["m10"] / M["m00"])
    cY = int(M["m01"] / M["m00"])
    Y.append(cY)
    print(cX, cY)
    plt.plot(cX, cY, 'ro')

print(np.argsort(Y))

In [163]:
# get ellipse of that random frame
frame = cv2.imread(os.path.join(save_path, "frame799.jpg"))
ellipse, x_center, y_center, x_horizontal, y_horizontal, x_vertical, y_vertical = get_plot_features(frame, threshold = 31)

plt.figure(figsize = (10,10))
ax = plt.gca()
ax.imshow(frame)
ax.plot(x_horizontal, y_horizontal, linestyle = '--', linewidth = 3, color = 'orange')
ax.plot(x_vertical, y_vertical, linestyle = '--', linewidth = 3, color = 'orange')
ax.scatter(x_center, y_center, s = 200,  marker = 'x', color = 'orange')
ax.plot(ellipse[:,0], ellipse[:,1], linewidth = 2.5, color = 'orange')
ax.fill(ellipse[:,0], ellipse[:,1], alpha = 0.15, color = 'orange')
ax.set_title('Random frame', fontsize = 20)
# set ticks and labels to 12
plt.xticks(fontsize = 12)
plt.yticks(fontsize = 12)

plt.show()


# Importing stimulus to get the real orientation

In [120]:
# generate a vector in which every component is the time of the frame
# find matlab file in calibration folder
mat_file = [f for f in os.listdir(calibration_folder) if f.endswith('.mat')][0]
container = io.loadmat(os.path.join(calibration_folder, mat_file))

n_repeats = container['stim_data']['n_repeats'][0][0][0][0]
distance_mat = container['stim_data']['distance'][0][0][0]
leg_1_start = container['stim_data']['leg_1_start'][0][0][0] * 1000
leg_1_end = np.sort(container['stim_data']['leg_1_end'][0][0][0]) * 1000
off_time = np.sort(container['stim_data']['off_time'][0][0][0]) * 1000
encoder_time = container['stim_data']['encoder_time'][0][0][0]
encoder_time = encoder_time - encoder_time[0]
encoder_angle = container['stim_data']['encoder_angle'][0][0][0]

frames_time = np.linspace(encoder_time[0],encoder_time[-1], 1498)
encoder_angle_interp = np.interp(frames_time, encoder_time, encoder_angle)

plt.plot(frames_time, encoder_angle_interp)
plt.axvline(leg_1_start[0] + 2500, color = 'r')
plt.axvline(leg_1_end[0], color = 'r')
plt.axvline(leg_1_start[1] + 2500, color = 'r')
plt.axvline(leg_1_end[1], color = 'r')
plt.axvline(leg_1_start[2] + 2500, color = 'r')
plt.axvline(leg_1_end[2], color = 'r')
plt.axvline(leg_1_start[3] + 2500, color = 'r')
plt.axvline(leg_1_end[3], color = 'r')
plt.axvline(leg_1_start[4] + 2500, color = 'r')
plt.axvline(leg_1_end[4], color = 'r')


In [121]:
def get_distance_frames(distance = 1):
    
    repeat_start = leg_1_start[distance] + 2500
    repeat_end = leg_1_end[distance]
    
    repeat_frames = (frames_time > repeat_start) & (frames_time < repeat_end)
    repeat_frames = np.where(repeat_frames)[0]
    
    for i in range(len(repeat_frames)):
        repeat_frames[i] -= 10
    
    return repeat_frames

# Read the frames

In [122]:
images_path = os.listdir(save_path)
images_path.sort()

def get_image(frame_number):
    # find the image in images_path that ends with str(frame_number) + '.jpg'
    image_name = [f for f in images_path if f.endswith('frame' + str(frame_number) + '.jpg')][0]
    return cv2.imread(os.path.join(save_path, image_name))

In [168]:
# distances_dict = {}
colors =['orange', '#35b779', '#31688e', '#440154']

for distance in [3]:
    distance_frames = get_distance_frames(distance = distance)
    
    entries = {}
    for i in range(len(distance_frames)):
        
        #print in a single line and update it the number of the frame
        print('Distance ' + str(distance+1) + ' - Frame ' + str(distance_frames[i]), end = '\r')
        
        _, features = get_features_of_ellipse(get_image(distance_frames[i]), threshold = 40)
        
        ellipse, x_center, y_center, x_horizontal, y_horizontal, x_vertical, y_vertical = get_plot_features(get_image(distance_frames[i]), threshold = 40)
        
        # fig, ax = plt.subplots()
        # ax.imshow(get_image(distance_frames[i]))
        # ax.plot(x_horizontal, y_horizontal, linestyle = '--', linewidth = 3, color = colors[distance])
        # ax.plot(x_vertical, y_vertical, linestyle = '--', linewidth = 3, color = colors[distance])
        # ax.scatter(x_center, y_center, s = 200,  marker = 'x', color = colors[distance])
        # ax.plot(ellipse[:,0], ellipse[:,1], linewidth = 3, color = colors[distance])
        # ax.fill(ellipse[:,0], ellipse[:,1], alpha = 0.2, color = colors[distance])
        # ax.set_title('Distance ' + str(distance+1) + ' - Frame ' + str(distance_frames[i]), fontsize = 20)
        # plt.savefig(calibration_folder +  r'/distance-' + str(distance+1) + '/frame_' + str(distance_frames[i]) + '.png')
        # plt.close()
        
        if features is not None:
            entries[i] = features

    df = pd.DataFrame(columns= features.keys())
    
    for entry_name, features in entries.items():
        df.loc[entry_name] = features
        
    distances_dict[distance] = df

In [188]:
fig, ax = plt.subplots(1,2, figsize = (15,4))

for distance in [0, 1, 2, 3]:
    distance_frames = get_distance_frames(distance = distance)
    ax[0].plot(encoder_angle_interp[distance_frames], distances_dict[distance]['x_center'], color = colors[distance], label = 'Distance ' + str(distance+1), linewidth = 2)
    ax[0].set_title('X center', fontsize = 20)
    ax[1].plot(encoder_angle_interp[distance_frames], distances_dict[distance]['y_center'], color = colors[distance], label = 'Distance ' + str(distance+1), linewidth = 2)
    ax[1].set_title('Y center', fontsize = 20)

for axi in ax:
    axi.legend()
    axi.set_xlabel('Angle', fontsize = 15)
    axi.set_ylabel('Pixel #', fontsize = 15)
    axi.tick_params(axis='both', which='major', labelsize=12)


In [177]:
fig, ax = plt.subplots(1,2, figsize = (15,4))
# add some space between title and plot

for distance in [0, 1, 2, 3]:
    distance_frames = get_distance_frames(distance = distance)
    ax[1].plot(encoder_angle_interp[distance_frames], distances_dict[distance]['bright_median'], color = colors[distance], label = 'Distance ' + str(distance+1), linewidth = 2)
    ax[1].set_title('White Part', fontsize = 20)
    ax[0].plot(encoder_angle_interp[distance_frames], distances_dict[distance]['dark_median'], color = colors[distance], label = 'Distance ' + str(distance+1), linewidth = 2)
    ax[0].set_title('Black Part', fontsize = 20)

for axi in ax:
    axi.legend()
    axi.set_xlabel('Angle', fontsize = 15)
    axi.set_ylabel('Intesity', fontsize = 15)
    axi.tick_params(axis='both', which='major', labelsize=12)

# Add some space between title and plot
plt.subplots_adjust(top=0.85)

In [186]:
total_area = get_image(0).shape[0] * get_image(0).shape[1]
print(total_area)

plt.figure()
for distance in [0, 1, 2, 3]:
    distance_frames = get_distance_frames(distance = distance)
    plt.plot(encoder_angle_interp[distance_frames], 100 * distances_dict[distance]['area_region']/total_area, color = colors[distance], label = 'Distance ' + str(distance+1), linewidth = 2)
    plt.title('Stimulus total size', fontsize = 20)
    plt.xlabel('Angle', fontsize = 15)
    plt.ylabel('Area (%)', fontsize = 15)
    plt.tick_params(axis='both', which='major', labelsize=12)
    plt.legend()


In [196]:
print(distance_frames)

In [ ]:
# importing movie py libraries
from moviepy.editor import VideoClip
from moviepy.video.io.bindings import mplfig_to_npimage
%matplotlib inline

# duration of the video
distance = 0
distance_frames = get_distance_frames(distance = distance)
duration = len(distance_frames)
 
# matplot subplot
fig, ax = plt.subplots()
 
# method to get frames
def make_frame(t):
    print(t)
     
    # clear
    ax.clear()
     
    # plotting line
    ax.imshow(get_image(int(distance_frames[int(t)])), cmap = 'gray')
    xy_ellipse, _ = get_features_of_ellipse(get_image(int(distance_frames[int(t)])), threshold = 31)
    if xy_ellipse is not None:
        ax.plot(xy_ellipse[:,0], xy_ellipse[:,1], color = colors[distance])
        ax.fill(xy_ellipse[:,0], xy_ellipse[:,1], color = colors[distance], alpha = 0.1)
     
    # returning numpy image
    return mplfig_to_npimage(fig)
 
# creating animation
animation = VideoClip(make_frame, duration = duration)
 
# # displaying animation with auto play and looping
animation.ipython_display(fps = 1, loop = False, autoplay = True)

# Calibration from webcam

In [207]:
import cv2
vc = open_camera(1)

In [192]:
frame = get_frame(vc, show=True)

In [183]:
close_camera(vc)

In [211]:
ref_frame = get_frame(vc, show=False)
threshold = 70
get_contour(ref_frame, threshold = threshold, show=True)
ref_values = get_points_enclosed_by_ellipse(ref_frame, threshold = threshold, quadrant = None, show=True)
_, ref_features = get_features_of_ellipse(ref_frame, threshold = threshold)

plt.figure()
plt.hist(ref_values, bins = np.linspace(0, 255, 100))

In [212]:
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

frame = get_frame(vc, show = False)
frame = get_frame(vc, show = False)
total_area = frame.shape[0] * frame.shape[1]
values = get_points_enclosed_by_ellipse(frame, threshold = threshold, quadrant = None)

ref_ellipse, ref_x_center, ref_y_center, ref_x_horizontal, ref_y_horizontal, ref_x_vertical, ref_y_vertical = get_plot_features(ref_frame, threshold = threshold)
ellipse, x_center, y_center, x_horizontal, y_horizontal, x_vertical, y_vertical = get_plot_features(frame, threshold = threshold)
_, features = get_features_of_ellipse(frame, threshold = 60)

fig,ax = plt.subplots(2, 1, gridspec_kw={'height_ratios': [3, 1]}, figsize = (10,10))

# add title and space between title and top subplot
fig.suptitle('Calibration', fontsize = 20)
fig.subplots_adjust(top=0.95)

# add more space between plots
fig.subplots_adjust(bottom=0.35)

# add space inbetween
fig.subplots_adjust(hspace=0.2)

# for the actual frame
ax[0].imshow(frame)
ax[0].plot(x_horizontal, y_horizontal, linewidth = 1, color = 'blue')
ax[0].plot(x_vertical, y_vertical, linewidth = 1, color = 'blue')
ax[0].scatter(x_center, y_center, s = 200,  marker = 'x', color = 'blue')
ax[0].plot(ellipse[:,0], ellipse[:,1], linewidth = 2.5, color = 'blue')
ax[0].fill(ellipse[:,0], ellipse[:,1], alpha = 0.1, color = 'blue')

# add to the text the x, y position
text_1 = 'X, Y: ' + str(round(features['x_center'], 2)) + ', ' + str(round(features['y_center'], 2))
text_2 = 'Area: '  + str(round(features['area_region']/total_area, 4))

text_3 = 'X, Y: ' + str(round(ref_features['x_center'], 2)) + ', ' + str(round(ref_features['y_center'], 2))
text_4 = 'Area: '  + str(round(ref_features['area_region']/total_area, 4))

# add legend with text_1 and text_2
patch_1 = mlines.Line2D([], [], color='blue', marker='x', markersize=10, label=text_1)
patch_2 = mpatches.Patch(color='blue', label=text_2)
patch_3  = mlines.Line2D([], [], color='orange', marker='x', markersize=10, label=text_3)
patch_4 = mpatches.Patch(color='orange', label=text_4)

ax[0].legend(handles=[patch_1, patch_2, patch_3, patch_4])

# for the reference
ax[0].plot(ref_ellipse[:,0], ref_ellipse[:,1], linewidth = 2.5, color = 'orange')
ax[0].fill(ref_ellipse[:,0], ref_ellipse[:,1], alpha = 0.1, color = 'orange')
ax[0].scatter(ref_x_center, ref_y_center, s = 200,  marker = 'x', color = 'orange')
ax[0].plot(ref_x_horizontal, ref_y_horizontal, linewidth = 1, color = 'orange')
ax[0].plot(ref_x_vertical, ref_y_vertical, linewidth = 1, color = 'orange')

# set ticks and labels to 12
plt.xticks(fontsize = 12)
plt.yticks(fontsize = 12)

ax[1].hist(values, bins=np.linspace(0,255,100), color = 'blue', alpha = 0.5, label = 'Actual')
ax[1].hist(ref_values, bins=np.linspace(0,255,100), color = 'orange', alpha = 0.5, label = 'Reference')

ax[1].legend()
ax[1].set_xlabel('Intensity', fontsize = 15)
ax[1].set_ylabel('Frequency', fontsize = 15)
ax[1].tick_params(axis='both', which='major', labelsize=12)

plt.show()

In [143]:
close_camera(vc)